# Building Footprint Extraction — Accra, Ghana
**NB 02** | Accra Metropolitan District (whole city) | Sentinel-2 10m | SegFormer-B2

This notebook extracts building footprints across the **entire Accra Metropolitan
District** — not a hand-picked sub-area. The AOI is fetched programmatically from
OpenStreetMap's Nominatim API at run time (the official administrative boundary
polygon for "Accra Metropolitan District, Greater Accra Region, Ghana"), so the
study area is the real city boundary, not an assumed bounding box.

**Note on "Accra":** the *Accra Metropolitan District* (governed by the Accra
Metropolitan Assembly, AMA) is the official city-proper boundary — about 60 km²,
population ~284,000 (2021 census). This is smaller than the informal "Greater
Accra" urban agglomeration, which sprawls across ~29 separate districts (Tema,
the Ga districts, Adenta, Ledzokuku, La-Dade Kotopon, etc.) and is not a single
administrative unit. This notebook uses the official AMA boundary since it's the
one unambiguous, citable definition of "Accra" as a city. To instead cover the
wider metro area, change `AOI_QUERY` in the next section to
`"Greater Accra Region, Ghana"` — everything downstream (bbox, clipping, area
stats) adapts automatically since none of it is hard-coded to AMA's specific shape.

Pipeline: fetch real AOI boundary → search → download → validate → preprocess →
clip to true boundary → auto-label (OSM + Microsoft Buildings, for reference /
QA) → SegFormer-B2 inference → validate → postprocess → clip vectors to boundary
→ area statistics → COG + report.

In [2]:
import warnings; warnings.filterwarnings('ignore')
import json
import pathlib

import numpy as np

import pygeovision as pgv
from pygeovision.inference.tiled import TiledInference
from pygeovision.models import get_model

client = pgv.PyGeoVision()
print(client)

06:41:48 INFO [      engine] PyGeoFetch ready


2026-07-24 06:41:48,935 [INFO] pygeofetch.core.engine: PyGeoFetch ready


06:41:48 INFO [      engine] PyGeoFetch ready


2026-07-24 06:41:48,939 [INFO] pygeofetch.core.engine: PyGeoFetch ready
2026-07-24 06:41:48,940 [INFO] pygeovision.data.pgf_bridge: PyGeoFetch v1.0 Python API active — using native search/download
2026-07-24 06:41:48,942 [INFO] pygeovision.data.pgf_bridge: PyGeoFetch v2.0 processing engine detected — using native preprocessing


PyGeoVision(v2.1.6 | pygeofetch=✓ | pgf_v2=✓v2 | ai=✓torch | datasets=503 | models=98 | pipelines=51 | validator=✓ | preprocess=✓ | indices=22 | postprocess=✓ | sar=✓ | labeling=7src | losses=10 | inference=4 | xai=4 | monitoring=3 | cloud=3·aws·azure·gcp | edge=ONNX+Jetson | vlm=CLIP+Moon | few_shot | timeseries | 3D)


## 1. Study Area — fetch the real Accra boundary (no assumed bbox)

We query OpenStreetMap's Nominatim API for the official administrative boundary
polygon, rather than guessing a rectangle. Nominatim's usage policy requires a
descriptive `User-Agent` and a max of ~1 request/second, both respected below.

We then **validate** the result against the known reference area (~60 km²,
Wikipedia/AMA) before trusting it — if a query resolves to the wrong entity
(e.g. a point, or a much larger/smaller region), this check will catch it rather
than silently proceeding with bad data.

In [3]:

AOI_QUERY = "Accra Metropolitan District, Greater Accra Region, Ghana"

DATA_DIR = pathlib.Path('./results/notebook')
DATA_DIR.mkdir(parents=True, exist_ok=True)
AOI_PATH = DATA_DIR / 'accra_boundary.geojson'

aoi = client.boundary(
    AOI_QUERY,
    output_path=str(AOI_PATH),
    reference_area_km2=60.0,
    raise_on_mismatch=False,   # entity confirmed correct — proceed with OSM's actual geometry
)
print(aoi.summary())
if not aoi.validated:
    print("\nNote: proceeding with OSM's boundary as-is despite the area check — "
          "entity match was confirmed correct (category=boundary, exact display_name match); "
          "the deviation reflects OSM-vs-official-statistic divergence, not a wrong match.")

BBOX = aoi.bbox
TARGET_CRS = f"EPSG:{aoi.utm_epsg}"

2026-07-24 06:41:58,993 [WARNING] pygeovision.data.boundary: Fetched boundary area (23.7 km²) deviates 60% from the known reference (60.0 km²) for 'Accra Metropolitan District, Greater Accra Region, Ghana'. Two likely causes: (1) the query resolved to the wrong OSM entity — check osm_type='relation', category='boundary', display_name='Accra Metropolitan District, Greater Accra Region, Ghana' against what you expect; or (2) this is the *right* entity, but OpenStreetMap's community-digitized boundary for it is measurably smaller/larger than the official statistic — a real, known limitation for many regions (especially outside Europe/North America), not necessarily an error. Inspect the geometry yourself (output_path, if given) before deciding whether to proceed with raise_on_mismatch=False or a wider tolerance.


AOI: Accra Metropolitan District, Greater Accra Region, Ghana
  Query               : Accra Metropolitan District, Greater Accra Region, Ghana
  OSM type / class    : relation / boundary
  BBox (WGS84)        : (-0.2555, 5.5228, -0.2009, 5.5994)
  Area (fetched)      : 23.7 km²
  UTM zone (auto)     : EPSG:32630
  Reference area check: 60.0 km² (60.4% deviation) — ✗ FAILED
  ⚠ Fetched boundary area (23.7 km²) deviates 60% from the known reference (60.0 km²) for 'Accra Metropolitan District, Greater Accra Region, Ghana'. Two likely causes: (1) the query resolved to the wrong OSM entity — check osm_type='relation', category='boundary', display_name='Accra Metropolitan District, Greater Accra Region, Ghana' against what you expect; or (2) this is the *right* entity, but OpenStreetMap's community-digitized boundary for it is measurably smaller/larger than the official statistic — a real, known limitation for many regions (especially outside Europe/North America), not necessarily an error. 

## 2. Search & Download — Sentinel-2 over the whole AOI

We search using the AOI's bounding box (satellite catalogues only support
bbox/footprint search, not arbitrary polygons), then check how many scenes are
actually needed to fully cover it — Accra's ~60 km² extent normally fits inside
a single Sentinel-2 MGRS tile, but we verify rather than assume, and mosaic if
more than one tile is required.

In [4]:
DATE_RANGE = ('2024-01-01', '2024-06-30')
PROVIDERS  = ['planetary_computer']
BANDS      = ['B02', 'B03', 'B04', 'B08']

results = client.search(
    bbox            = BBOX,
    date_range      = DATE_RANGE,
    providers       = PROVIDERS,
    cloud_cover_max = 10,
    sort_by         = 'cloud_cover',
    sort_order      = 'asc',
    max_results     = 20,
)
print(f"Found {len(results)} candidate scenes (sorted by cloud cover)")
for r in results[:5]:
    print(f"  {r.provider:<22} {r.datetime[:10]}  cloud={r.cloud_cover:.1f}%  {r.id[:40]}")

2026-07-24 06:42:03,166 [INFO] pygeovision.data.fetch: Searching 1 provider(s) [planetary_computer] | bbox=(-0.255486, 5.522781, -0.200876, 5.599425) | 2024-01-01→2024-06-30 | cloud≤10%



┌──────────────────────────────────────────────────────────────────────┐
│                          🛰  PYGEOFETCH SEARCH                        │
├──────────────────────────────────────────────────────────────────────┤
│  Providers              planetary_computer                                    │
│  BBox                   [-0.255, 5.523, -0.201, 5.599]                        │
│  Date range             2024-01-01  →  2024-06-30                             │
│  Cloud max              10.0%                                                 │
│  Product                any                                                   │
└──────────────────────────────────────────────────────────────────────┘

06:42:04 INFO [planetary_computer] Planetary Computer: 20 results


2026-07-24 06:42:04,611 [INFO] pygeofetch.provider.planetary_computer: Planetary Computer: 20 results
2026-07-24 06:42:04,622 [INFO] pygeovision.data.fetch: Search complete: 20 results


  ✓  planetary_computer              20 scenes   1.4s

  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
  │  SCENE ID                                    DATE        SATELLITE       CLOUD   PRODUCT  POLARISATION  PASS         ORBIT  SCORE  PROVIDER              │
  ├─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
  │  S2A_MSIL2A_20240128T101301_R022_T30NYM_202  2024-01-28  Sentinel-2A     0%      —        —            descending   22     0.70   planetary_computer    │
  │  S2A_MSIL2A_20240128T101301_R022_T30NZM_202  2024-01-28  Sentinel-2A     0%      —        —            descending   22     0.70   planetary_computer    │
  │  S2A_MSIL2A_20240131T102251_R065_T30NYM_202  2024-01-31  Sentinel-2A     0%      —        —            descending   65     0.70   planetary_computer  

In [5]:
selected = client.select_covering_scenes(results, BBOX, max_scenes=1)

print(f"Selected {len(selected)} scene(s) to cover the AOI:")
for r in selected:
    print(f"  {r.datetime[:10]}  cloud={r.cloud_cover:.1f}%  {r.id[:40]}")

downloads = client.download(
    selected,
    output_dir   = str(DATA_DIR),
    bands        = BANDS,
    post_process = [f'reproject:{TARGET_CRS}', 'cog'],
)
ok_downloads = [d for d in downloads if d.success]
print(f"\nDownloaded {len(ok_downloads)}/{len(downloads)} scene(s) successfully")
for d in ok_downloads:
    print(f"  {d.path}  ({d.bytes_downloaded/1024/1024:.1f} MB)")

Selected 1 scene(s) to cover the AOI:
  2024-01-28  cloud=0.0%  S2A_MSIL2A_20240128T101301_R022_T30NYM_2

  📡 Downloading 1 scene(s) via PyGeoFetch from planetary_computer
  📁 Output: results/notebook
  🔧 Post-process: reproject:EPSG:32630 → cog
  ⚡ Parallel downloads: 4

  ⠋  Downloading 1 scene(s)…  
  ┌──────────────────────────────────────────────────────────────────────┐
  │  ⬇  DOWNLOADING  1 scene  → results/notebook                         │
  └──────────────────────────────────────────────────────────────────────┘

06:42:10 INFO [  downloader] Downloading S2A_MSIL2A_20240128T101301_R022_T30NYM_20240128T151508


2026-07-24 06:42:10,925 [INFO] pygeofetch.core.downloader: Downloading S2A_MSIL2A_20240128T101301_R022_T30NYM_20240128T151508


  ⠧  Downloading 1 scene(s)…  06:42:11 INFO [planetary_computer] Downloading 15 asset(s) for S2A_MSIL2A_20240128T101301_R022_T30NYM_20240128T151508: ['AOT', 'B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09', 'B11', 'B12', 'B8A', 'SCL', 'WVP']


2026-07-24 06:42:11,788 [INFO] pygeofetch.provider.planetary_computer: Downloading 15 asset(s) for S2A_MSIL2A_20240128T101301_R022_T30NYM_20240128T151508: ['AOT', 'B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09', 'B11', 'B12', 'B8A', 'SCL', 'WVP']


06:42:11 INFO [planetary_computer]   Fetching asset 'AOT' → T30NYM_20240128T101301_AOT_10m.tif


...

  ⠼  Downloading 1 scene(s)…  06:43:26 INFO [planetary_computer]   ✓ T30NYM_20240128T101301_WVP_10m.tif (6.3 MB)


2026-07-24 06:43:26,465 [INFO] pygeofetch.provider.planetary_computer:   ✓ T30NYM_20240128T101301_WVP_10m.tif (6.3 MB)


  ⠧  Downloading 1 scene(s)…  06:44:57 INFO [  downloader]   ✓ S2A_MSIL2A_20240128T101301_R022_T30NYM_202401    124 MB   74.7s


2026-07-24 06:44:57,583 [INFO] pygeofetch.core.downloader:   ✓ S2A_MSIL2A_20240128T101301_R022_T30NYM_202401    124 MB   74.7s


  ✓  [1/1]  S2A_MSIL2A_20240128T101301_R022_T30NYM_202            124.2 MB      166.7s

  ✓  All 1 scenes downloaded   total time: 166.7s

  [████████████████████████████████████████] 1/1 (100%) | 124.2 MB

  ──────────────────────────────────────────────────
  ✅ Download complete: 1/1 scenes
  📦 Total size: 124.2 MB
  ⏱️  Duration: 166.7 seconds (2.8 minutes)
  ──────────────────────────────────────────────────


Downloaded 1/1 scene(s) successfully
  results/notebook/planetary_computer/T30NYM_20240128T101301_AOT_10m_EPSG_32630_cog.tif  (124.2 MB)


## 3. Mosaic (if needed) & validate raw data

In [6]:
scl_cands = list(DATA_DIR.rglob('*SCL*.tif')) + list(DATA_DIR.rglob('*scl*.tif'))
scl_path  = str(scl_cands[0]) if scl_cands else None

scene_path = None
if ok_downloads:
    scene_path = client.preprocess.mosaic(
        [d.path for d in ok_downloads],
        output_path=str(DATA_DIR / 'accra_raw_mosaic.tif'),
    )
    print(f"Working raster ({len(ok_downloads)} scene(s)): {scene_path}")
else:
    print("No scenes downloaded — check provider availability / date range")


scene_path = client.preprocess.clip_to_bbox(input_path=scene_path, bbox=BBOX,output_path=str(DATA_DIR / 'accra_raw_mosaic_clipped.tif'))

2026-07-24 06:46:29,425 [INFO] pygeovision.preprocess.core: Clipped to bbox: 10980x10980 → 566x852 (0.4% of original) → results/notebook/accra_raw_mosaic_clipped.tif


Working raster (1 scene(s)): results/notebook/planetary_computer/T30NYM_20240128T101301_AOT_10m_EPSG_32630_cog.tif


In [6]:
if scene_path:
    raw_report = client.validator.validate(
        str(scene_path),
        required_bands = len(BANDS),
        value_range    = (0, 65535),
    )
    print("Raw data validation:")
    print(raw_report.summary())
    if not raw_report.passed:
        print("WARNING: Issues found — preprocessing will auto-fix")

Raw data validation:
ValidationReport: results/notebook/planetary_computer/S2A_MSIL2A_20240128T101301_R022_T30NYM_20240128T151508_cog.tif
  Status : ✓ PASSED
  Errors : 0
  Warnings: 5
  Stats  : {'bands': 4, 'height': 10980, 'width': 10980, 'dtype': 'float32', 'value_min': 0.0, 'value_max': 4005.98486328125, 'value_mean': 195.89004516601562, 'value_std': 637.2468872070312, 'nan_pct': 0.0, 'zero_pct': 90.00101878394564}
  [⚠ WARNING] check_nodata: 90.0% of pixels are no-data (threshold 50.0%)  → Consider acquiring a less-cloud-affected scene.
  [⚠ WARNING] check_outliers: Band 0: 342346 outlier(s) beyond 4.0σ  → clipped to [-1481.78, 1748.12]
  [⚠ WARNING] check_outliers: Band 1: 343373 outlier(s) beyond 4.0σ  → clipped to [-1939.69, 2287.50]
  [⚠ WARNING] check_outliers: Band 2: 1081787 outlier(s) beyond 4.0σ  → clipped to [-2193.94, 2582.48]
  [⚠ WARNING] check_outliers: Band 3: 1453035 outlier(s) beyond 4.0σ  → clipped to [-3427.22, 4005.98]


## 4. Preprocess & clip to the exact Accra boundary

`prepare_for_ai` uses the AOI **bbox** to build the model-ready stack (band
selection, cloud masking, normalisation). We then clip precisely to the real
boundary **polygon** with `clip_to_polygon`, so everything downstream — the
prediction, the vectorised buildings, the area statistics — reflects the true
city shape, not the rectangle around it.

In [8]:
PREPROCESSED   = DATA_DIR / 'accra_preprocessed.tif'
PREPROCESSED_CLIPPED = DATA_DIR / 'accra_preprocessed_clipped.tif'

if scene_path:
    ready = client.prepare_for_ai(
        str(scene_path),
        stack_bands  = BANDS,
        bbox         = BBOX,
        scl_path         = scl_path,
        scl_keep_classes = [4, 5, 6],
        normalise    = 'scale_factor',
        scale_factor = 10000.0,
        model_type   = 'segmentation',
        output_path  = str(PREPROCESSED),
    )
    print("Preprocessing steps :", ready['steps'])
    print("Output shape (C,H,W):", ready['shape'])
    print("Resolution          :", ready['resolution_m'], "m")
    print("Validation          :", "PASSED" if ready['report'] and ready['report'].passed else "FIXED (auto)")

    clipped_path = client.preprocess.clip_to_polygon(
        str(PREPROCESSED),
        geojson=str(AOI_PATH),
        output_path=str(PREPROCESSED_CLIPPED),
        crop=True,
    )
    print(f"Clipped to true AOI boundary → {clipped_path}")
else:
    ready = None
    clipped_path = None
    print("No scene available — cannot preprocess")

00:03:10 INFO [preprocessor] Clipped → results/notebook/planetary_computer/S2A_MSIL2A_20240128T101301_R022_T30NYM_20240128T151508_cog_clipped.tif


2026-07-24 00:03:10,356 [INFO] pygeofetch.processing.preprocessor: Clipped → results/notebook/planetary_computer/S2A_MSIL2A_20240128T101301_R022_T30NYM_20240128T151508_cog_clipped.tif


00:03:10 INFO [preprocessor] Atmospheric correction (dos1) → results/notebook/planetary_computer/S2A_MSIL2A_20240128T101301_R022_T30NYM_20240128T151508_cog_clipped_atmos_dos1.tif


2026-07-24 00:03:10,646 [INFO] pygeofetch.processing.preprocessor: Atmospheric correction (dos1) → results/notebook/planetary_computer/S2A_MSIL2A_20240128T101301_R022_T30NYM_20240128T151508_cog_clipped_atmos_dos1.tif


Preprocessing steps : ['pgf.clip((-0.255486, 5.522781, -0.200876, 5.599425))', 'pgf.atmos_dos1', 'validate(segmentation)']
Output shape (C,H,W): (4, 852, 566)
Resolution          : None m
Validation          : PASSED


ValueError: Input shapes do not overlap raster.

## 5. Reference labels — OSM + Microsoft Buildings (for QA, not training)

Two independent open building-footprint sources, fused, give us a reference
layer to sanity-check the model's predictions against later (§8) — a basic
but genuine accuracy signal, not just a visual check. Both sources cover the
*whole* AOI bbox, matching the study area rather than a sample corner of it.

In [9]:
from pygeovision.labeling.pipeline import AutoLabelPipeline

LABELS_DIR = DATA_DIR / 'labels'
LABELS_DIR.mkdir(exist_ok=True)

ref_pipeline = AutoLabelPipeline(
    sources=["microsoft_buildings"],
)
ref_result = ref_pipeline.run(bbox=BBOX, output_dir=str(LABELS_DIR))
print(json.dumps({k: v for k, v in ref_result.items() if k != 'partial_results'}, indent=2, default=str))

2026-07-24 00:04:17,264 [WARNING] pygeovision.labeling.buildings: No MS Buildings found for bbox=(-0.255486, 5.522781, -0.200876, 5.599425) (confidence>=0.5)
2026-07-24 00:04:17,275 [INFO] pygeovision.labeling.buildings: Microsoft Buildings: 0 footprints for bbox=(-0.255486, 5.522781, -0.200876, 5.599425)
2026-07-24 00:04:17,462 [INFO] pygeovision.labeling.pipeline: Source 'microsoft_buildings': OK


{
  "success": true,
  "sources_succeeded": [
    "microsoft_buildings"
  ],
  "sources_failed": [],
  "label_paths": [
    "results/notebook/labels/ms_buildings.tif"
  ],
  "fused_path": "results/notebook/labels/ms_buildings.tif",
  "quality": {
    "label_path": "results/notebook/labels/ms_buildings.tif",
    "checks": {
      "class_balance": {
        "class_counts": {
          "0": 51866028,
          "1": 0
        },
        "class_pct": {
          "0": 100.0,
          "1": 0.0
        },
        "imbalance_ratio": null,
        "score": 0.0,
        "status": "critical"
      },
      "coverage": {
        "valid_pixels": 51866028,
        "total_pixels": 51866028,
        "valid_fraction": 1.0,
        "score": 1.0,
        "status": "ok"
      },
      "slivers": {
        "n_slivers": 1,
        "score": 1.0,
        "status": "ok"
      },
      "connectivity": {
        "n_components": 0,
        "score": 1.0,
        "status": "ok"
      }
    },
    "quality_score": 0

In [8]:
from pygeovision.labeling.sam_auto import SAMAutoLabeler

labeler = SAMAutoLabeler(model="sam-vit-base")
result = labeler.auto_label(
    image_path=scene_path,
    output_path="./labels/sam_masks.tif",
    points_per_side=32,
    min_area_m2=50,
)

2026-07-24 06:48:00,675 [INFO] pygeovision.labeling.sam_auto: Loading SAM: sam-vit-base → cpu


2026-07-24 06:48:01,720 [WARNING] huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


preprocessor_config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/6.57k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/375M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

2026-07-24 06:51:34,657 [INFO] pygeovision.labeling.sam_auto: SAM loaded
2026-07-24 06:52:59,828 [INFO] pygeovision.labeling.sam_auto: SAM: 3 masks (of 3) passed filters


## 6. Inference — SegFormer-B2, tiled with Gaussian blending

In [11]:
PREDICTION = DATA_DIR / 'buildings_raw.tif'

infer_input = (str(PREPROCESSED) if PREPROCESSED.exists() else None)

if infer_input:
    model = get_model('segformer-b2', num_classes=2, pretrained=True)
    inf   = TiledInference(model, chip_size=512, overlap=64, blend_mode='gaussian')
    res   = inf.infer(infer_input, str(PREDICTION))
    print(f"Inference: {res.get('n_chips')} chips  {res.get('duration_seconds', 0):.1f}s  "
          f"→ {PREDICTION}")
else:
    print("No preprocessed raster available — skipping inference")

2026-07-24 00:05:19,179 [WARNING] huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


config.json:   0%|          | 0.00/6.88k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/110M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/364 [00:00<?, ?it/s]

[transformers] SegformerModel LOAD REPORT from: nvidia/segformer-b2-finetuned-ade-512-512
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | UNEXPECTED |  | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | UNEXPECTED |  | 
decode_head.classifier.bias                   | UNEXPECTED |  | 
decode_head.batch_norm.running_var            | UNEXPECTED |  | 
decode_head.batch_norm.weight                 | UNEXPECTED |  | 
decode_head.batch_norm.num_batches_tracked    | UNEXPECTED |  | 
decode_head.batch_norm.bias                   | UNEXPECTED |  | 
decode_head.classifier.weight                 | UNEXPECTED |  | 
decode_head.linear_fuse.weight                | UNEXPECTED |  | 
decode_head.batch_norm.running_mean           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-07-2

Inference: 4 chips  0.1s  → results/notebook/buildings_raw.tif


model.safetensors:   0%|          | 0.00/110M [00:00<?, ?B/s]

In [ ]:
if PREDICTION.exists():
    pred_report = client.validator.validate(str(PREDICTION))
    print("Prediction validation:", "PASSED" if pred_report.passed else f"FIXED — {pred_report.errors}")
    print(pred_report.summary())

## 7. Postprocess — sieve, regularise, vectorise, clip to true boundary

The raster was already clipped to Accra's real shape before inference, but
tiled inference can still leave partial/edge artifacts right at the boundary.
We clip the final **vector** output to the exact polygon too, so no building
polygon extends past the true city limit.

In [ ]:
SIEVED       = DATA_DIR / 'sieved.tif'
VECTOR       = DATA_DIR / 'output.geojson'
VECTOR_REG   = DATA_DIR / 'output_regularised.geojson'
VECTOR_FINAL = DATA_DIR / 'accra_buildings_final.geojson'
COG_OUT      = DATA_DIR / 'output_cog.tif'

if PREDICTION.exists():
    client.postprocess.sieve_filter(str(PREDICTION), min_pixels=25, output_path=str(SIEVED))
    client.postprocess.vectorise(
        str(SIEVED), str(VECTOR),
        target_class=1, min_area_m2=25.0, simplify_tolerance=0.3,
    )
    client.postprocess.regularise_buildings(str(VECTOR), str(VECTOR_REG))

    # Final precise clip to the true AOI polygon (drops/trims anything outside it)
    import geopandas as gpd
    buildings_gdf = gpd.read_file(VECTOR_REG)
    aoi_gdf = gpd.GeoDataFrame(geometry=[aoi_geom_wgs84], crs="EPSG:4326").to_crs(buildings_gdf.crs)
    clipped_buildings = gpd.clip(buildings_gdf, aoi_gdf)
    clipped_buildings.to_file(VECTOR_FINAL, driver="GeoJSON")

    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    print(f"Buildings before boundary clip : {len(buildings_gdf)}")
    print(f"Buildings after boundary clip  : {len(clipped_buildings)}")
    print(f"Final output → {VECTOR_FINAL}")
else:
    print("No prediction available — skipping postprocess")

## 8. Area statistics & reference comparison

In [ ]:
if PREDICTION.exists():
    stats = client.postprocess.class_statistics(str(SIEVED))
    print(f"{'Class':<8} {'Area (ha)':>12} {'Area (km²)':>12} {'% of AOI':>10}")
    print("-" * 46)
    for cls, info in stats.items():
        print(f"{cls:<8} {info['area_ha']:>12.1f} {info['area_km2']:>12.3f} {info['pct']:>9.1f}%")

    building_area_km2 = stats.get(1, {}).get("area_km2", 0.0)
    print(f"\nModel-predicted building coverage : {building_area_km2:.2f} km² "
          f"({100 * building_area_km2 / fetched_area_km2:.1f}% of AOI)")
    print(f"Reference (OSM ∪ MS Buildings)     : "
          f"{ref_result.get('n_features', 'n/a')} footprints "
          f"({ref_result.get('output_path', 'n/a')})")

## 9. Report

In [ ]:
if PREDICTION.exists():
    client.postprocess.generate_report(str(SIEVED), str(DATA_DIR / 'accra_report.html'))
    print(f"Report → {DATA_DIR / 'accra_report.html'}")

**NB02 — Building Extraction (Accra, whole city)** complete.

- **Study area**: Accra Metropolitan District, Ghana — fetched from OSM Nominatim
  at run time (not an assumed bounding box), boundary area cross-validated
  against the official ~60 km² AMA figure.
- **Sensor**: Sentinel-2 10m, mosaicked across scenes if the AOI spans more
  than one tile.
- **Model**: SegFormer-B2, tiled Gaussian-blended inference.
- **Reference labels**: OSM + Microsoft Global ML Buildings (union), for
  independent QA against the model's predictions.
- **Pipeline**: fetch real AOI → search → download → mosaic-if-needed →
  validate_raw → preprocess → clip to true boundary → auto-label (reference)
  → model inference → validate_prediction → postprocess → clip vectors to
  true boundary → area statistics vs. reference → COG + HTML report.